# 01 - Data exploration

Audit the raw Kaggle dataset: class balance, duplicates, unreadable files, image sizes, and a visual spot-check for label noise.

Put the dataset in `data/raw/<class>/` first (see `docs/DATASET_GUIDE.md`).

**Running on Google Colab?** Upload or clone the project, then run once:

```python
from google.colab import drive
drive.mount("/content/drive")          # optional: keep data and models on Drive
%cd /content/iClaim
%pip install -q -r requirements.txt
```
Use a GPU runtime (Runtime > Change runtime type) for training.

In [ ]:
# --- Setup (works locally and in Google Colab) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.config import load_config, resolve_path, set_seed
cfg = load_config()
set_seed(cfg["project"]["seed"])
print("Project root:", ROOT)

## Audit

In [ ]:
from src.data_loader import audit_dataset, list_raw_images

audit = audit_dataset(cfg)
audit

## Class balance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(data=audit, x="class", y="images", ax=ax, color="#2a78d6")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
ax.set_xlabel(""); ax.set_ylabel("Images"); ax.set_title("Images per class")
plt.xticks(rotation=20); plt.tight_layout()
(resolve_path(cfg["paths"]["figures_dir"])).mkdir(parents=True, exist_ok=True)
plt.savefig(resolve_path(cfg["paths"]["figures_dir"]) / "class_distribution.png", dpi=150)
plt.show()

total = audit["images"].sum()
if total:
    ratio = audit["images"].max() / max(audit["images"][audit["images"] > 0].min(), 1)
    print(f"Imbalance ratio: {ratio:.1f}:1  (use class weights if > 2)")

## Sample images per class

Look for mislabeled or irrelevant images. Note anything unusual for the report.

In [ ]:
from PIL import Image
import random

images = list_raw_images(resolve_path(cfg["paths"]["raw_data"]), cfg)
classes = [c for c, v in images.items() if v]
n = 4
fig, axes = plt.subplots(len(classes), n, figsize=(3 * n, 2.4 * max(len(classes), 1)), squeeze=False)
rng = random.Random(cfg["project"]["seed"])
for r, cls in enumerate(classes):
    picks = rng.sample(images[cls], min(n, len(images[cls])))
    for c in range(n):
        ax = axes[r][c]; ax.axis("off")
        if c < len(picks):
            ax.imshow(Image.open(picks[c]).convert("RGB"))
            if c == 0:
                ax.set_title(cls, loc="left", fontsize=10)
plt.tight_layout()
plt.savefig(resolve_path(cfg["paths"]["figures_dir"]) / "sample_images.png", dpi=120)
plt.show()

## Image sizes

In [ ]:
sizes = []
for cls, paths in images.items():
    for p in paths[:300]:
        try:
            with Image.open(p) as im:
                sizes.append({"class": cls, "width": im.size[0], "height": im.size[1]})
        except Exception:
            pass
sz = pd.DataFrame(sizes)
if not sz.empty:
    display(sz.describe().round(0))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(sz["width"], sz["height"], s=8, alpha=0.4, color="#2a78d6")
    ax.set_xlabel("Width (px)"); ax.set_ylabel("Height (px)"); ax.set_title("Image dimensions")
    plt.tight_layout(); plt.show()

## Notes for the report

- Dataset name / URL / licence:
- Class imbalance:
- Label noise observed:
- Other quality issues: